# Milestone 2: Data Quality dan Profiling

Lanjutkan dari hasil `01_data_profiling.ipynb`. Notebook ini menerapkan profiling DuckDB `SUMMARIZE` dan transformasi Polars untuk menangani missing values, outlier, duplikat, serta validasi data dasar. Sesuaikan nama kolom dan aturan validasi dengan dataset masing-masing.

In [1]:
from pathlib import Path
import duckdb
import polars as pl

cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd.parent if cwd.name == "notebooks" else cwd
DATA_PATH = PROJECT_ROOT / "data" / "raw" / "stocks.csv"
OUTPUT_PATH = PROJECT_ROOT / "data" / "processed" / "dataset_bersih.parquet"

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset tidak ditemukan: {DATA_PATH}. "
        "Pastikan stocks.csv sudah diekstrak ke data/raw/."
    )

df = pl.read_csv(DATA_PATH)
print(f"Ukuran awal: {df.shape[0]:,} baris x {df.shape[1]} kolom")
df.head()

Ukuran awal: 2,061,655 baris x 7 kolom


datetime,symbol,open,high,low,close,volume
str,str,f64,f64,f64,f64,f64
"""2005-06-30 02:00:00""","""IDX:AALI""",3111.92455,3131.620275,3111.92455,3131.620275,1.580926e6
"""2005-07-01 02:00:00""","""IDX:AALI""",3111.92455,3111.92455,3072.5331,3092.228825,901210.0
"""2005-07-04 02:00:00""","""IDX:AALI""",3072.5331,3072.5331,2993.7502,3033.14165,3.226588e6
"""2005-07-05 02:00:00""","""IDX:AALI""",3033.14165,3033.14165,3013.445925,3013.445925,1.928718e6
"""2005-07-06 02:00:00""","""IDX:AALI""",3033.14165,3072.5331,3033.14165,3052.837375,1.632333e6


## 1. Profiling dengan DuckDB `SUMMARIZE`

Dapatkan ringkasan statistik seluruh kolom (tipe data, null%, min/max, distinct count) dalam satu perintah.

In [2]:
connection = duckdb.connect()
connection.register("dataset", df)

cursor = connection.execute("SUMMARIZE SELECT * FROM dataset")
cols = [desc[0] for desc in cursor.description]
profil = pl.DataFrame(cursor.fetchall(), schema=cols, orient="row")
profil

column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
str,str,str,str,i64,str,str,str,str,str,i64,"decimal[38,2]"
"""datetime""","""VARCHAR""","""1990-04-23 02:00:00""","""2025-07-04 02:00:00""",8442,null,null,null,null,null,2061655,0.00
"""symbol""","""VARCHAR""","""IDX:AADI""","""IDX:ZYRX""",932,null,null,null,null,null,2061655,0.00
"""open""","""DOUBLE""","""1.0""","""226150.0""",76603,"""1193.5359202012473""","""3601.599418546406""","""119.48212540038215""","""314.3702496457439""","""909.395380195211""",2061655,0.00
"""high""","""DOUBLE""","""1.0""","""226150.0""",76074,"""1213.0925827229078""","""3651.3229033488265""","""122.83434458621323""","""320.4493217230142""","""927.5396493014551""",2061655,0.00
"""low""","""DOUBLE""","""1.0""","""226150.0""",76985,"""1174.24299314465""","""3550.6578236076393""","""116.79671101136078""","""307.08952744279094""","""894.9553759139394""",2061655,0.00
"""close""","""DOUBLE""","""1.0""","""226150.0""",76781,"""1193.8202564397618""","""3599.772669965685""","""119.44945834455356""","""313.2059047735494""","""911.8916512397095""",2061655,0.00
"""volume""","""DOUBLE""","""0.0""","""45941762000.0""",672876,"""20306576.36255158""","""134989236.5755948""","""53127.31487159818""","""937960.4804630183""","""8740489.130469706""",2061655,0.00


## 2. Deteksi Missing Values

Lihat jumlah dan persentase null per kolom sebelum memutuskan strategi penanganan.

In [3]:
jumlah_baris = len(df)
null_summary = (
    df.null_count()
    .transpose(include_header=True, header_name="kolom", column_names=["jumlah_null"])
    .with_columns(
        (pl.col("jumlah_null") / jumlah_baris * 100).round(2).alias("persen_null")
    )
    .sort("jumlah_null", descending=True)
)
null_summary

kolom,jumlah_null,persen_null
str,u32,f64
"""datetime""",0,0.0
"""symbol""",0,0.0
"""open""",0,0.0
"""high""",0,0.0
"""low""",0,0.0
"""close""",0,0.0
"""volume""",0,0.0


## 3. Penanganan Missing Values

Ganti `kolom_numerik` dan `kolom_kategorik` dengan nama kolom asli pada dataset. Pilih strategi sesuai jenis data: median/mean untuk numerik, nilai default atau modus untuk kategorik.

In [4]:
df_missing_handled = df.with_columns([
    pl.col("symbol").cast(pl.String),
    pl.col("datetime").cast(pl.String),
    pl.col("open").cast(pl.Float64),
    pl.col("high").cast(pl.Float64),
    pl.col("low").cast(pl.Float64),
    pl.col("close").cast(pl.Float64),
    pl.col("volume").cast(pl.Float64),
]).drop_nulls(subset=["symbol", "datetime", "close"])

print(f"Null sebelum: {df.null_count().sum_horizontal().sum():,}")
print(f"Null sesudah: {df_missing_handled.null_count().sum_horizontal().sum():,}")

Null sebelum: 0
Null sesudah: 0


## 4. Deteksi dan Penanganan Outlier

Gunakan metode IQR sebagai titik awal, lalu kombinasikan dengan batas bisnis (domain knowledge) yang relevan dengan dataset Anda.

In [5]:
KOLOM_NUMERIK = "close"

q1 = df_missing_handled[KOLOM_NUMERIK].quantile(0.25)
q3 = df_missing_handled[KOLOM_NUMERIK].quantile(0.75)
iqr = q3 - q1
batas_bawah = max(0.0, q1 - 1.5 * iqr)
batas_atas = q3 + 1.5 * iqr

print(f"Q1={q1}, Q3={q3}, IQR={iqr}")
print(f"Batas bawah: {batas_bawah}, Batas atas: {batas_atas}")

# Jika ingin tetap mempertahankan nilai harga asli emiten IDX (karena saham bluechip memang jauh di atas Q3 + 1.5*IQR),
# cukup pastikan harga valid (> 0) atau gunakan df_outlier_handled sesuai kebutuhan analisis.
df_outlier_handled = df_missing_handled.filter(pl.col("close") > 0)

Q1=120.0, Q3=910.0, IQR=790.0
Batas bawah: 0.0, Batas atas: 2095.0


## 5. Deduplikasi

Tentukan kolom kunci bisnis (misalnya ID unik) sebelum menghapus duplikat — jangan deduplikasi berdasarkan seluruh baris.

In [6]:
KUNCI_UNIK = ["symbol", "datetime"]

total_baris = len(df_outlier_handled)
df_dedup = df_outlier_handled.unique(
    subset=KUNCI_UNIK,
    keep="first",
    maintain_order=True,
)
print(f"Total baris: {total_baris:,}")
print(f"Baris unik berdasarkan {KUNCI_UNIK}: {len(df_dedup):,}")
print(f"Duplikat terhapus: {total_baris - len(df_dedup):,}")

Total baris: 2,061,655
Baris unik berdasarkan ['symbol', 'datetime']: 2,061,655
Duplikat terhapus: 0


## 6. Validasi Data Dasar

Tulis beberapa aturan validasi sederhana untuk memastikan data bersih memenuhi batas yang masuk akal. Gantikan contoh aturan di bawah dengan aturan yang relevan dengan dataset Anda.

In [7]:
connection.register("dataset_bersih", df_dedup)

validasi = connection.execute("""
    SELECT
        COUNT(*) AS total_baris,
        SUM(CASE WHEN close IS NULL OR symbol IS NULL OR datetime IS NULL THEN 1 ELSE 0 END) AS pelanggaran_null,
        SUM(CASE WHEN close <= 0 OR high < low OR close < low OR close > high THEN 1 ELSE 0 END) AS pelanggaran_harga_ohlc
    FROM dataset_bersih
""").fetchall()

pl.DataFrame(
    validasi,
    schema=["total_baris", "pelanggaran_null", "pelanggaran_harga_ohlc"],
    orient="row",
)

total_baris,pelanggaran_null,pelanggaran_harga_ohlc
i64,i64,i64
2061655,0,0


## 7. Simpan Data Bersih

Simpan hasil ke `data/processed/` agar dapat dipakai pada Milestone 3 tanpa mengulang proses cleaning.

In [8]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
df_dedup.write_parquet(OUTPUT_PATH)

print(f"Baris sebelum cleaning: {len(df):,}")
print(f"Baris sesudah cleaning: {len(df_dedup):,}")
print(f"Data bersih tersimpan di: {OUTPUT_PATH}")

Baris sebelum cleaning: 2,061,655
Baris sesudah cleaning: 2,061,655
Data bersih tersimpan di: C:\Users\ACER\tugas1-abd-2026\data\processed\dataset_bersih.parquet


## Catatan Milestone 2

Dokumentasikan di sini: dimensi kualitas data yang paling bermasalah, strategi yang dipilih untuk missing values dan outlier, jumlah baris yang berubah, serta aturan validasi yang diterapkan.

In [ ]:
Dimensi kualitas data yang dicek: missing values (null), kevalidan harga dan struktur OHLC (open, high, low, close, volume), duplikasi kunci ["symbol", "datetime"], serta konsistensi tipe data.

Strategi missing values & anomali: menyeragamkan tipe data kolom, membuang baris yang tidak lengkap pada kolom inti (symbol, datetime, close), serta menyaring nilai harga yang tidak valid (close <= 0 atau ketidaksesuaian batas OHLC).

Deduplikasi & perubahan baris: menghapus baris duplikat berdasarkan pasangan kunci ["symbol", "datetime"] (keep="first"), lalu membandingkan jumlah baris sebelum dan sesudah cleaning.

Validasi & penyimpanan: memverifikasi tidak ada pelanggaran null maupun anomali harga pada dataset_bersih, kemudian menyimpan hasil akhir ke data/processed/dataset_bersih.parquet untuk digunakan pada Milestone 3.